# Preparación de Datos — GEIH 2024
**Entorno:** Local — scikit-learn + category-encoders + joblib

---
### Setup y carga del dataset

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from category_encoders import TargetEncoder

# Rutas locales
BASE    = Path('parquet')
ENTRADA = BASE / 'geih_2024_crudo.parquet'
SALIDA  = BASE

df = pd.read_parquet(ENTRADA)

print('Dataset cargado')
print(f'   Filas:    {len(df):,}')
print(f'   Columnas: {df.shape[1]}')
print(f'   Memoria:  {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

Dataset cargado
   Filas:    358,029
   Columnas: 21
   Memoria:  57.4 MB


---
### Excluir registros sin variable objetivo

Eliminamos los pensionados (P6920 = 3) y los nulos en `INFORMAL` porque no aportan a la variable objetivo: un trabajador activo en riesgo de informalidad.

In [2]:
antes = len(df)

df = df[df['INFORMAL'].notna()].copy()

excluidos = antes - len(df)
print(f'Registros excluidos (INFORMAL = NaN): {excluidos:,}  ({excluidos/antes*100:.1f}%)')
print(f'Registros para modelado:              {len(df):,}')
print(f'  Informales (1): {df["INFORMAL"].sum():,.0f}  ({df["INFORMAL"].mean()*100:.1f}%)')
print(f'  Formales   (0): {(df["INFORMAL"]==0).sum():,.0f}  ({(df["INFORMAL"]==0).mean()*100:.1f}%)')

Registros excluidos (INFORMAL = NaN): 6,308  (1.8%)
Registros para modelado:              351,721
  Informales (1): 200,796  (57.1%)
  Formales   (0): 150,925  (42.9%)


---
### Nulos en predictores

Estrategia general:
- Variables numéricas con pocos nulos → imputar con la **mediana**
- Variables categóricas con pocos nulos → imputar con la **moda**
- Variables con muchos nulos (> 30%) → evaluar si se incluy

In [3]:
PREDICTORES_RAW = [
    'P3271',     # Sexo
    'P6040',     # Edad
    'P3042',     # Nivel educativo
    'P3042S1',   # Años en el nivel
    'P6070',     # Estado civil
    'DPTO',      # Departamento
    'CLASE',     # Zona (cabecera / rural)
    'P6430',     # Posición ocupacional
    'P6450',     # Tipo de contrato
    'P6460',     # Naturaleza del contrato
    'P6800',     # Horas trabajadas
    'P3069',     # Tamaño del establecimiento
    'RAMA2D_R4', # Rama de actividad
    'P7040',     # Pluriempleo
]

# Reporte de nulos en predictores
nulos = (
    df[PREDICTORES_RAW].isna().sum()
    .to_frame('n_nulos')
    .assign(pct=lambda x: (x['n_nulos'] / len(df) * 100).round(2))
    .sort_values('pct', ascending=False)
)

print('Nulos en predictores:')
print(nulos.to_string())

Nulos en predictores:
           n_nulos    pct
P6460       213708  60.76
P6450       139385  39.63
P3042            0   0.00
P3042S1          0   0.00
P3271            0   0.00
P6040            0   0.00
DPTO             0   0.00
P6070            0   0.00
P6430            0   0.00
CLASE            0   0.00
P6800            0   0.00
P3069            0   0.00
RAMA2D_R4        0   0.00
P7040            0   0.00


---
### Ingeniería de variables

In [4]:
# ── Grupos de edad (curva en U — EDA Celda 12) ───────────────────────
# Mínimo de informalidad en 25-34; sube drásticamente después de 64
bins_edad   = [14, 24, 34, 44, 54, 64, 120]
labels_edad = ["15-24", "25-34", "35-44", "45-54", "55-64", "65+"]
df["EDAD_GRUPO"] = pd.cut(
    df["P6040"], bins=bins_edad, labels=labels_edad
).astype(str)

# ── Años de educación continuos (relación negativa clara — EDA Celda 8) ─
mapa_edu = {
     1:  0,   # Ninguno
     2:  1,   # Preescolar
     3:  3,   # Primaria incompleta
     4:  5,   # Primaria completa
     5:  7,   # Secundaria incompleta
     6:  9,   # Secundaria completa
     7: 10,   # Media incompleta
     8: 11,   # Media completa
     9: 14,   # Técnica / Tecnológica
    10: 16,   # Universitaria
    11: 17,   # Especialización
    12: 18,   # Maestría
    13: 21,   # Doctorado
}
df["ANIOS_EDU"] = (
    df["P3042"].map(mapa_edu).fillna(0) +
    df["P3042S1"].fillna(0).clip(0, 6)
)

# ── Indicadores binarios ──────────────────────────────────────────────

# Cuenta propia: P6430 == 4 (EDA: tasa 85.4%)
df["CUENTA_PROPIA"] = (df["P6430"] == 4).astype(int)

# Microempresa: 
# Tamaño Empresa = 1 (1 persona), 2 (2-5), 3 (6-10)
df["MICROEMPRESA"] = (df["P3069"].isin([1, 2, 3])).astype(int)

# Subempleo por horas: menos de 32 horas semanales
df["SUBEMPLEADO"] = (df["P6800"] < 32).astype(int)

# Pluriempleo: recodificar P7040 de (1=Sí, 2=No) a (1=Sí, 0=No)
df["PLURIEMPLEO"] = df["P7040"].map({1: 1, 2: 0})

# Contrato verbal: P6450 == 1
# Nota: P6450 tiene 39.79% nulos — se conserva con imputación por moda
# Supuesto: trabajadores sin dato de contrato tienden a ser informales;
# la moda (contrato verbal) es estructuralmente coherente con ese perfil
df["CONTRATO_VERBAL"] = (df["P6450"] == 1).astype(int)


print("Ingeniería de variables completada")
nuevas = ["EDAD_GRUPO","ANIOS_EDU","CUENTA_PROPIA","MICROEMPRESA",
          "SUBEMPLEADO","PLURIEMPLEO","CONTRATO_VERBAL"]
for v in nuevas:
    print(f"   {v}: {df[v].nunique()} valores únicos, {df[v].isna().sum()} nulos")


Ingeniería de variables completada
   EDAD_GRUPO: 6 valores únicos, 0 nulos
   ANIOS_EDU: 28 valores únicos, 0 nulos
   CUENTA_PROPIA: 2 valores únicos, 0 nulos
   MICROEMPRESA: 2 valores únicos, 0 nulos
   SUBEMPLEADO: 2 valores únicos, 0 nulos
   PLURIEMPLEO: 2 valores únicos, 0 nulos
   CONTRATO_VERBAL: 2 valores únicos, 0 nulos


---
### Definir features y variable objetivo

Selección final de columnas para el modelo.
Las variables excluidas están documentadas con su justificación.

In [5]:
# Variable objetivo
TARGET = "INFORMAL"

col_geografica = "DPTO"

FEATURES = [
    # Demográficas
    "P3271",          # Sexo
    "P6040",          # Edad (continua)
    "EDAD_GRUPO",     # Edad en grupos (no lineal — curva en U)
    "ANIOS_EDU",      # Años de educación continuos
    "P6070",          # Estado civil
    "CLASE",          # Zona urbana / rural (EDA: 48.9% vs 83.8%)
    col_geografica,   # DPTO (target encoding)

    # Laborales originales
    "P6430",          # Posición ocupacional
    "P6450",          # Tipo de contrato (40% nulos — imputar con moda)
    "P6800",          # Horas trabajadas
    "P3069",          # Tamaño del establecimiento (correlación -0.78)
    "RAMA2D_R4",      # Rama de actividad (target encoding)

    # Features derivadas
    "CUENTA_PROPIA",
    "MICROEMPRESA",
    "SUBEMPLEADO",
    "PLURIEMPLEO",
    "CONTRATO_VERBAL",
]

EXCLUIDAS = {
    "INGLABO":   "Data leakage — consecuencia de la informalidad, no causa",
    "P6920":     "Fuente de la variable objetivo",
    "FEX_C18":   "Factor de expansión — no es un predictor",
    "P3042":     "Reemplazada por ANIOS_EDU (versión continua)",
    "P3042S1":   "Incorporada en ANIOS_EDU",
    "P7040":     "Reemplazada por PLURIEMPLEO (recodificada)",
    "P6460":     "60.78% de nulos — demasiado alto para imputar de forma válida",
}

print(f"Features seleccionadas: {len(FEATURES)}")
for f in FEATURES:
    print(f"  {f}")
print(f"Variables excluidas: {len(EXCLUIDAS)}")
for var, razon in EXCLUIDAS.items():
    print(f"  {var}: {razon}")


Features seleccionadas: 17
  P3271
  P6040
  EDAD_GRUPO
  ANIOS_EDU
  P6070
  CLASE
  DPTO
  P6430
  P6450
  P6800
  P3069
  RAMA2D_R4
  CUENTA_PROPIA
  MICROEMPRESA
  SUBEMPLEADO
  PLURIEMPLEO
  CONTRATO_VERBAL
Variables excluidas: 7
  INGLABO: Data leakage — consecuencia de la informalidad, no causa
  P6920: Fuente de la variable objetivo
  FEX_C18: Factor de expansión — no es un predictor
  P3042: Reemplazada por ANIOS_EDU (versión continua)
  P3042S1: Incorporada en ANIOS_EDU
  P7040: Reemplazada por PLURIEMPLEO (recodificada)
  P6460: 60.78% de nulos — demasiado alto para imputar de forma válida


---
### División train / test

Dividimos **antes** de construir el pipeline para que los parámetros
de transformación (medias, encodings) se calculen solo con datos de entrenamiento.

- **80% entrenamiento** — para ajustar el modelo
- **20% test** — para evaluación final, no se toca hasta la Fase 5
- **`stratify=y`** — preserva la proporción de formales/informales en ambos conjuntos

In [6]:
X = df[FEATURES].copy()
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y        # mantiene proporción formal/informal
)

print(f'Train: {len(X_train):,} filas  ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test:  {len(X_test):,} filas  ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nProporción INFORMAL en train: {y_train.mean()*100:.1f}%')
print(f'Proporción INFORMAL en test:  {y_test.mean()*100:.1f}%')

Train: 281,376 filas  (80%)
Test:  70,345 filas  (20%)

Proporción INFORMAL en train: 57.1%
Proporción INFORMAL en test:  57.1%


---
### Clasificar features por tipo

El `ColumnTransformer` de scikit-learn aplica transformaciones diferentes
según el tipo de variable. Clasificamos las features antes de construirlo.

In [7]:
COLS_NUM = [
    "P6040",     # Edad
    "ANIOS_EDU", # Años de educación
    "P6800",     # Horas trabajadas
]

COLS_BIN = [
    "P3271",           # Sexo (1/2)
    "CLASE",           # Zona (1/2)
    "CUENTA_PROPIA",   # 0/1
    "MICROEMPRESA",    # 0/1
    "SUBEMPLEADO",     # 0/1
    "PLURIEMPLEO",     # 0/1
    "CONTRATO_VERBAL", # 0/1 (40% nulos imputados con moda)
]

# P6460 excluida: 60.78% nulos
COLS_OHE = [
    "EDAD_GRUPO",  # 6 categorías (curva en U)
    "P6070",       # Estado civil
    "P6430",       # Posición ocupacional
    "P6450",       # Tipo de contrato (40% nulos — moda)
    "P3069",       # Tamaño del establecimiento
]

COLS_TARGET = ["RAMA2D_R4"]
COLS_TARGET.append("DPTO")

todas     = set(COLS_NUM + COLS_BIN + COLS_OHE + COLS_TARGET)
faltantes = set(FEATURES) - todas
sobrantes = todas - set(FEATURES)

print("Features por tipo de transformación:")
print(f"  Numéricas (impute+scale):        {COLS_NUM}")
print(f"  Binarias (solo impute):          {COLS_BIN}")
print(f"  Categóricas OHE:                 {COLS_OHE}")
print(f"  Alta cardinalidad (target enc.): {COLS_TARGET}")
print()
if faltantes:
    print(f"Sin transformación asignada: {faltantes}")
elif sobrantes:
    print(f"⚠️  En listas pero no en FEATURES: {sobrantes}")
else:
    print("Todas las features tienen transformación asignada")


Features por tipo de transformación:
  Numéricas (impute+scale):        ['P6040', 'ANIOS_EDU', 'P6800']
  Binarias (solo impute):          ['P3271', 'CLASE', 'CUENTA_PROPIA', 'MICROEMPRESA', 'SUBEMPLEADO', 'PLURIEMPLEO', 'CONTRATO_VERBAL']
  Categóricas OHE:                 ['EDAD_GRUPO', 'P6070', 'P6430', 'P6450', 'P3069']
  Alta cardinalidad (target enc.): ['RAMA2D_R4', 'DPTO']

Todas las features tienen transformación asignada


---
### Pipeline de preprocesamiento

El `ColumnTransformer` aplica transformaciones distintas a cada grupo
de columnas en paralelo y las concatena en una sola matriz numérica.

```
X_train (raw)
    ├─ Numéricas    → Imputar mediana → StandardScaler
    ├─ Binarias     → Imputar moda
    ├─ OHE          → Imputar moda → OneHotEncoder
    └─ Alta card.   → TargetEncoder (usa y_train internamente)
         ↓
X_train (procesada) — lista para el modelo
```

In [8]:
# Sub-pipelines por tipo de variable

pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

pipe_bin = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
])

pipe_ohe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    )),
])

# TargetEncoder: smoothing=1.0 suaviza categorías con pocos registros
pipe_target = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder(smoothing=1.0)),
])

# ColumnTransformer: combina todos los sub-pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num',    pipe_num,    COLS_NUM),
        ('bin',    pipe_bin,    COLS_BIN),
        ('ohe',    pipe_ohe,    COLS_OHE),
        ('target', pipe_target, COLS_TARGET),
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

print('Preprocessor construido')
print('   (Aún no ajustado — se ajusta en la siguiente celda)')

Preprocessor construido
   (Aún no ajustado — se ajusta en la siguiente celda)


---
### Ajustar el pipeline y transformar los datos

> ⚠️ `fit_transform` solo sobre `X_train`.
> `X_test` se transforma con `transform` (sin reajustar).

In [9]:
# Ajustar sobre train y transformar ambos conjuntos
# El TargetEncoder necesita y_train para calcular las medias por categoría
X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc  = preprocessor.transform(X_test)

# Recuperar nombres de columnas resultantes
try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    feature_names = [f'feat_{i}' for i in range(X_train_proc.shape[1])]

print(f'Pipeline ajustado y aplicado')
print(f'   X_train procesada: {X_train_proc.shape}')
print(f'   X_test  procesada: {X_test_proc.shape}')
print(f'   Features resultantes: {len(feature_names)}')
print(f'\nPrimeras 10 features: {list(feature_names[:10])}')

Pipeline ajustado y aplicado
   X_train procesada: (281376, 45)
   X_test  procesada: (70345, 45)
   Features resultantes: 45

Primeras 10 features: ['P6040', 'ANIOS_EDU', 'P6800', 'P3271', 'CLASE', 'CUENTA_PROPIA', 'MICROEMPRESA', 'SUBEMPLEADO', 'PLURIEMPLEO', 'CONTRATO_VERBAL']


---
### Guardar datasets procesados y pipeline

In [10]:
import joblib

# Convertir a DataFrames con nombres de columnas
df_train = pd.DataFrame(X_train_proc, columns=feature_names)
df_train['INFORMAL'] = y_train.values

df_test = pd.DataFrame(X_test_proc, columns=feature_names)
df_test['INFORMAL'] = y_test.values

# Guardar en Parquet (localmente en parquet/)
ruta_train = SALIDA / 'train.parquet'
ruta_test  = SALIDA / 'test.parquet'
ruta_pipe  = SALIDA / 'preprocessor.joblib'

df_train.to_parquet(ruta_train, index=False)
df_test.to_parquet(ruta_test,   index=False)
joblib.dump(preprocessor, ruta_pipe)

print('Archivos guardados en parquet/:')
for ruta in [ruta_train, ruta_test, ruta_pipe]:
    mb = ruta.stat().st_size / (1024 * 1024)
    print(f'   {ruta.name:<30} {mb:.1f} MB')

print('\nResumen Final')
print(f'  Train: {len(df_train):,} filas × {df_train.shape[1]-1} features')
print(f'  Test:  {len(df_test):,} filas × {df_test.shape[1]-1} features')
print('\nPróximo paso → Modelado (Fase 4)')

Archivos guardados en parquet/:
   train.parquet                  2.4 MB
   test.parquet                   0.6 MB
   preprocessor.joblib            0.0 MB

Resumen Final
  Train: 281,376 filas × 45 features
  Test:  70,345 filas × 45 features

Próximo paso → Modelado (Fase 4)
